In [2]:
import pandas as pd
import numpy as np
import optuna
from optuna.samplers import TPESampler
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score

# ============================================================================
# 1. LOAD DATA & FEATURE ENGINEERING
# ============================================================================
try:
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')
    print("✅ Loaded Data.")
except FileNotFoundError:
    raise FileNotFoundError("❌ Files not found.")

def create_advanced_features(df):
    df = df.copy()
    # Reusing your winning feature set
    activity_cols = ['hobby_engagement_level', 'physical_activity_index', 
                     'creative_expression_index', 'altruism_score']
    df['total_activity'] = df[activity_cols].sum(axis=1)
    df['support_guidance_combo'] = df['support_environment_score'] * (df['external_guidance_usage'] + 1)
    df['focus_efficiency'] = df['focus_intensity'] / (df['consistency_score'] + 1)
    df['consistency_gap'] = 30 - df['consistency_score']
    df['focus_sq'] = df['focus_intensity'] ** 2
    df['focus_X_consistency'] = df['focus_intensity'] * df['consistency_score']
    df['low_focus_high_consist'] = ((df['focus_intensity'] < 5) & (df['consistency_score'] > 24)).astype(int)
    return df

train_df = create_advanced_features(train_df)
test_df = create_advanced_features(test_df)

X = train_df.drop(['participant_id', 'personality_cluster'], axis=1)
y = train_df['personality_cluster']
test_ids = test_df['participant_id']
X_test = test_df.drop(['participant_id'], axis=1)

# Encode Target
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# ============================================================================
# 2. PREPROCESSING DEFINITION
# ============================================================================
cat_cols = [
    'identity_code', 'cultural_background', 'age_group', 
    'upbringing_influence', 'support_environment_score', 
    'hobby_engagement_level', 'physical_activity_index',
    'creative_expression_index', 'altruism_score',
    'low_focus_high_consist'
]
num_cols = [c for c in X.columns if c not in cat_cols]

# Define the transformer (will be used inside the pipeline)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ]
)

# ============================================================================
# 3. OPTUNA OBJECTIVE
# ============================================================================
def objective(trial):
    # Hyperparameter Search Space
    # C: Regularization parameter. The strength of the regularization is inversely proportional to C.
    c_val = trial.suggest_float("C", 0.1, 100.0, log=True)
    
    # Kernel: The type of math used to draw the line
    kernel = trial.suggest_categorical("kernel", ["rbf", "poly", "sigmoid"])
    
    # Gamma: Kernel coefficient (how far the influence of a single training example reaches)
    gamma = trial.suggest_categorical("gamma", ["scale", "auto"])
    
    # Specific parameters for Poly/Sigmoid kernels
    degree = 3
    coef0 = 0.0
    if kernel == "poly":
        degree = trial.suggest_int("degree", 2, 5)
        coef0 = trial.suggest_float("coef0", 0.0, 5.0)
    elif kernel == "sigmoid":
        coef0 = trial.suggest_float("coef0", 0.0, 5.0)

    # Build Pipeline
    model = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', SVC(
            C=c_val,
            kernel=kernel,
            gamma=gamma,
            degree=degree,
            coef0=coef0,
            class_weight='balanced', # Essential
            probability=False,       # False for tuning speed (True needed only for final probas)
            random_state=42
        ))
    ])
    
    # Stratified K-Fold Cross Validation
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    f1_scores = []
    
    for train_idx, val_idx in skf.split(X, y_encoded):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y_encoded[train_idx], y_encoded[val_idx]
        
        model.fit(X_train, y_train)
        preds = model.predict(X_val)
        f1_scores.append(f1_score(y_val, preds, average='macro'))
        
    return np.mean(f1_scores)

# Run Optimization
print("Starting SVM Optuna Tuning...")
sampler = TPESampler(seed=42)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=50) # 50 is usually plenty for SVM

print("\n✅ BEST SVM PARAMS:")
print(study.best_params)
print(f"🏆 Best Macro F1: {study.best_value:.4f}")

# ============================================================================
# 4. FINAL TRAINING & SUBMISSION
# ============================================================================
print("\nRetraining Final Model with Best Params...")

# Reconstruct best parameters
best = study.best_params
final_kernel = best["kernel"]
final_degree = best.get("degree", 3)
final_coef0 = best.get("coef0", 0.0)

final_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', SVC(
        C=best["C"],
        kernel=final_kernel,
        gamma=best["gamma"],
        degree=final_degree,
        coef0=final_coef0,
        class_weight='balanced',
        probability=True,        # TRUE for final training to get probabilities
        random_state=42
    ))
])

# Train on Full Data
final_pipeline.fit(X, y_encoded)

# Predict on Test (Probabilities for Ensembling)
print("Generating Predictions...")
test_probs = final_pipeline.predict_proba(X_test)

# Save Probs
prob_df = pd.DataFrame(test_probs, columns=[f'prob_{i}' for i in range(5)])
prob_df['participant_id'] = test_ids.values
prob_df.to_csv('svm_optuna_probs.csv', index=False)
print("✅ Saved 'svm_optuna_probs.csv'")

# Generate Submission Labels
final_indices = np.argmax(test_probs, axis=1)
final_labels = le.inverse_transform(final_indices)

submission = pd.DataFrame({
    'participant_id': test_ids,
    'personality_cluster': final_labels
})
submission.to_csv('submission_svm_optuna.csv', index=False)
print("✅ Saved 'submission_svm_optuna.csv'")

C:\Users\hrush\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2025-11-26 11:58:30,267] A new study created in memory with name: no-name-4f16a8cb-2a5c-4cbe-a8bb-f5eb8e0c106a


✅ Loaded Data.
Starting SVM Optuna Tuning...


[I 2025-11-26 11:58:30,750] Trial 0 finished with value: 0.555524490915509 and parameters: {'C': 1.3292918943162166, 'kernel': 'rbf', 'gamma': 'scale'}. Best is trial 0 with value: 0.555524490915509.
[I 2025-11-26 11:58:31,558] Trial 1 finished with value: 0.5672344918523851 and parameters: {'C': 0.14936568554617632, 'kernel': 'rbf', 'gamma': 'auto'}. Best is trial 1 with value: 0.5672344918523851.
[I 2025-11-26 11:58:32,233] Trial 2 finished with value: 0.5381966020991259 and parameters: {'C': 31.428808908401084, 'kernel': 'rbf', 'gamma': 'auto'}. Best is trial 1 with value: 0.5672344918523851.
[I 2025-11-26 11:58:33,055] Trial 3 finished with value: 0.538396183744567 and parameters: {'C': 1.976218934028007, 'kernel': 'poly', 'gamma': 'auto', 'degree': 3, 'coef0': 3.925879806965068}. Best is trial 1 with value: 0.5672344918523851.
[I 2025-11-26 11:58:33,447] Trial 4 finished with value: 0.5471044260549839 and parameters: {'C': 0.3972110727381912, 'kernel': 'poly', 'gamma': 'scale', 'd


✅ BEST SVM PARAMS:
{'C': 0.3036215803939498, 'kernel': 'rbf', 'gamma': 'scale'}
🏆 Best Macro F1: 0.5778

Retraining Final Model with Best Params...
Generating Predictions...
✅ Saved 'svm_optuna_probs.csv'
✅ Saved 'submission_svm_optuna.csv'
